In [43]:
# Generate a Saltelli design CSV ready for WRF runs + later Sobol analysis
from SALib.sample import saltelli,sobol
import pandas as pd

# ==== CONFIGURE HERE ====
d = 5                 # number of parameters (x1..xd)
max_total = 1000      # total rows <= this (first-order); change to 2000, etc.
second_order = False  # True -> include second-order terms (bigger design)
seed = 2025           # fix for reproducibility; change/remove for a new draw
out_path = f"saltelli_d{d}_{'2nd' if second_order else '1st'}_le{max_total}.csv"
# ========================

# SALib "problem" in [0,1] for each parameter
problem = {
    "num_vars": d,
    "names": [f"x{i}" for i in range(1, d+1)],
    "bounds": [[0.0, 1.0]] * d
}

# Choose base N so total rows <= max_total (formula depends on first/second order)
denom = (2*d + 2) if second_order else (d + 2)
N = max(1, max_total // denom)
print(f"Chosen base N={N} to get total rows <= {max_total}")
# Build the stacked Saltelli matrix (A, B, AB(i) …)
#X = saltelli.sample(problem, N, calc_second_order=second_order) deprecaterd
X = sobol.sample(problem, N, calc_second_order=second_order, seed=seed)
# Wrap in a DataFrame, add a stable run_id to preserve order for later analysis
df = pd.DataFrame(X, columns=problem["names"])
#df.insert(0, "run_id", range(len(df)))   # keep this order when mapping results back

# Save
df.to_csv(out_path, index=False)

print(f"d={d}, second_order={second_order}, max_total={max_total}")
print(f"Base N={N} -> total_rows={len(df)} (expected {N*denom})")
print(f"Saved: {out_path}")
df.head()


Chosen base N=142 to get total rows <= 1000
d=5, second_order=False, max_total=1000
Base N=142 -> total_rows=994 (expected 994)
Saved: saltelli_d5_1st_le1000.csv


/home/jorge.gacitua/salidas/miniconda3/envs/wrf_python/lib/python3.12/site-packages/SALib/sample/sobol.py:136: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  base_sequence = qrng.random(N)


,x1,x2,x3,x4,x5
0,0.516084,0.937721,0.531489,0.733956,0.645453
1,0.116637,0.937721,0.531489,0.733956,0.645453
2,0.516084,0.734433,0.531489,0.733956,0.645453
3,0.516084,0.937721,0.175217,0.733956,0.645453
4,0.516084,0.937721,0.531489,0.080927,0.645453
